# The Visual Storyteller — Inference & Analysis

**Notebook 2 of 2.** Demonstrates the trained captioning models on **unseen test images**.

Contents:
0. Colab bootstrap (mount Drive, clone repo, unzip data) — **Colab only**
1. Setup & load artifacts (vocab + trained models)
2. The graded entry point: `generate_caption(image_path, model) -> str`
3. Demonstration on held-out test images
4. Success cases
5. Failure cases (diagnosed)
6. Attention visualisation
7. Baseline vs Transformer comparison table

> Run **Restart & Run All** on a clean runtime before submitting.

## 0. Colab bootstrap (Colab only)

Mounts Drive, clones the repo, installs requirements, and unzips the dataset. No-op locally.

In [ ]:
import os, sys, subprocess, shutil

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    REPO_DIR = '/content/visual-storyteller'
    if not os.path.isdir(REPO_DIR):
        subprocess.run(['git', 'clone', 'https://github.com/Andria-colab/visual-storyteller.git', REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)

    DEST = '/content/flickr8k'
    ZIP  = '/content/drive/MyDrive/visual_storyteller/caption_data.zip'
    def has_data(d): return os.path.isdir(f'{d}/Images') and os.path.isfile(f'{d}/captions.txt')
    if not has_data(DEST):
        os.makedirs(DEST, exist_ok=True)
        TMP = '/content/_extract'
        shutil.rmtree(TMP, ignore_errors=True); os.makedirs(TMP)
        subprocess.run(['unzip', '-o', ZIP, '-d', TMP], capture_output=True)
        src = TMP
        if not has_data(src):
            subs = [f'{TMP}/{d}' for d in os.listdir(TMP) if os.path.isdir(f'{TMP}/{d}')]
            src = next((d for d in subs if has_data(d)), src)
        for name in ('Images', 'captions.txt'):
            s = f'{src}/{name}'
            if os.path.exists(s) and not os.path.exists(f'{DEST}/{name}'):
                shutil.move(s, f'{DEST}/{name}')
        shutil.rmtree(TMP, ignore_errors=True)

    if has_data(DEST):
        print(f'OK — repo at {os.getcwd()}, {len(os.listdir(DEST+"/Images"))} images ready')
    else:
        print('WARNING: images not found — captions will fail. Check ZIP path:', ZIP)
else:
    print('local run — skipping bootstrap')

# Make src importable
REPO = os.getcwd() if IN_COLAB else (os.path.abspath('..') if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd())
if REPO not in sys.path:
    sys.path.insert(0, REPO)

## 1. Setup

In [ ]:
import torch
import matplotlib.pyplot as plt
from PIL import Image

from src.config import default_config
from src.data import Vocabulary, load_captions, references_for, make_splits, load_official_splits
from src.decode import generate_caption, greedy_decode, beam_search
from src import evaluate as E

try:
    import google.colab  # noqa: F401
    cfg = default_config(); cfg.use_colab_paths()
except ImportError:
    cfg = default_config()

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)
print('checkpoint dir:', cfg.checkpoint_dir)
print('vocab file    :', cfg.vocab_file)

vocab = Vocabulary.load(cfg.vocab_file)
print('vocab size:', len(vocab))

MODELS = {}
MODELS_READY = False
try:
    from src.model import EncoderDecoder
    from src.encoder import build_image_transform

    def load_model(ckpt_name, variant):
        model = EncoderDecoder(vocab_size=len(vocab), cfg=cfg, variant=variant).to(device)
        ckpt = torch.load(os.path.join(cfg.checkpoint_dir, ckpt_name), map_location=device)
        model.load_state_dict(ckpt['model'])
        model.eval()
        model.vocab = vocab
        model.device = device
        model.variant = variant
        model.image_transform = build_image_transform()
        model.beam_size = cfg.beam_size
        model.max_len = cfg.max_caption_len
        return model

    MODELS['baseline'] = load_model('baseline_best.pt', 'lstm')
    MODELS['transformer'] = load_model('transformer_best.pt', 'transformer')
    MODELS_READY = True
    print('loaded:', list(MODELS))
except (ImportError, FileNotFoundError) as e:
    print('[notice] models not available —', type(e).__name__, e)

## 3. The graded entry point

`generate_caption(image_path: str, model: any) -> str` — the exact signature the brief asks for. It loads the image, runs the frozen encoder, beam-searches, and detokenises to a sentence. Defined in `src/decode.py`.

In [ ]:
# Held-out test images — untouched during development.
captions = load_captions(cfg.captions_file)
splits = load_official_splits(cfg.data_dir) or make_splits(list(captions), seed=cfg.seed)
test_ids = splits['test']
test_refs = references_for(test_ids, captions)
print(f'{len(test_ids)} held-out test images')

def show(image_id, caps):
    """Display a test image with model captions + a reference."""
    path = os.path.join(cfg.images_dir, image_id)
    plt.figure(figsize=(5, 5)); plt.imshow(Image.open(path)); plt.axis('off')
    title = '\n'.join(f'{k}: {v}' for k, v in caps.items())
    ref = ' '.join(test_refs[image_id][0]) if test_refs.get(image_id) else ''
    plt.title(title + (f'\nref: {ref}' if ref else ''), fontsize=9, loc='left')
    plt.show()

def caption_all(image_id):
    path = os.path.join(cfg.images_dir, image_id)
    return {name: generate_caption(path, m) for name, m in MODELS.items()}

In [ ]:
if MODELS_READY:
    demo_id = test_ids[0]
    print('image:', demo_id)
    print(caption_all(demo_id))
else:
    print('skipped — models not loaded')

## 4. Success cases

~8 strong examples where the models produce accurate, fluent captions. Pick a spread of scenes (people, animals, action, outdoor/indoor).

In [ ]:
# Curate after a first pass over the test set; placeholder picks the first few.
SUCCESS_IDS = test_ids[:8]

if MODELS_READY:
    for img_id in SUCCESS_IDS:
        show(img_id, caption_all(img_id))
else:
    print('skipped — models not loaded')

## 5. Failure cases (diagnosed)

~5 cases where captions go wrong. For each, note the **failure type**: object hallucination, miscounting, colour/scene error, repetition, or over-generic caption. These diagnoses are high-value for the grade.

In [ ]:
# (image_id, diagnosis) — fill in after inspecting outputs.
FAILURE_CASES = [
    # ('1234567890_abc.jpg', 'object hallucination: invents a dog not in the image'),
    # ('...jpg', 'miscount: says two when there are three'),
]

if MODELS_READY:
    for img_id, diagnosis in FAILURE_CASES:
        caps = caption_all(img_id); caps['DIAGNOSIS'] = diagnosis
        show(img_id, caps)
else:
    print('skipped — models not loaded')

## 6. Attention visualisation

What is each decoder *looking at* as it generates each word? `src/viz.py` runs a greedy decode and overlays the per-word attention on the image — Bahdanau attention for the LSTM baseline, last-layer cross-attention for the Transformer. These maps are the qualitative payoff of the attention mechanisms and a good place to read off *why* a caption went right or wrong.

In [ ]:
from src import viz as VZ

if MODELS_READY:
    viz_id = SUCCESS_IDS[0]
    print('attention over:', viz_id)
    for name, model in MODELS.items():
        fig = VZ.visualize_attention(model, os.path.join(cfg.images_dir, viz_id))
        fig.suptitle(f'{name} — attention per generated word', y=1.02)
        plt.show()
else:
    print('skipped — models not loaded')

## 7. Baseline vs Transformer — quantitative comparison

Corpus BLEU-1..4 over the full held-out test set against all 5 references, for each model and decoding strategy. Goes into `reports/results.md` and the README.

In [ ]:
def evaluate_model(model, image_ids, use_beam=True):
    """Caption every test image, return {image_id: hyp_tokens}."""
    hyps = {}
    for img_id in image_ids:
        path = os.path.join(cfg.images_dir, img_id)
        feats = model.encoder(model.image_transform(
            Image.open(path).convert('RGB')).unsqueeze(0).to(device)).squeeze(0)
        if use_beam:
            ids = beam_search(model, feats, vocab, beam_size=cfg.beam_size,
                              max_len=cfg.max_caption_len, device=device)
        else:
            ids = greedy_decode(model, feats.unsqueeze(0), vocab,
                                max_len=cfg.max_caption_len, device=device)[0]
        hyps[img_id] = vocab.decode(ids)
    return hyps

if MODELS_READY:
    rows = {}
    for name, model in MODELS.items():
        for strat, beam in [('greedy', False), ('beam', True)]:
            hyps = evaluate_model(model, test_ids, use_beam=beam)
            rows[f'{name}-{strat}'] = E.score_bleu(hyps, test_refs)
    table = E.format_score_table(rows)
    print(table)
    os.makedirs(os.path.join(REPO, 'reports'), exist_ok=True)
    with open(os.path.join(REPO, 'reports', 'results.md'), 'w', encoding='utf-8') as f:
        f.write('# Results — BLEU on held-out test set\n\n' + table + '\n')
    print('\nwrote reports/results.md')
else:
    print('skipped — models not loaded')